# PatchCore TensorRT FP16 Export

**Steps:**
1. Enable GPU: `Runtime -> Change runtime type -> T4 GPU`
2. Run all cells top to bottom
3. Download `patchcore_cable.trt` and copy to your local `exports/` folder

> **Note:** a TensorRT engine only runs on the same TensorRT version and GPU architecture it was built with. An engine built here (T4, sm_75) will not load on e.g. an RTX 3080 or in the Docker image — rebuild with `src/optimize_trt.py` on the target machine.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tensorrt pycuda

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select exports/patchcore_cable.onnx
import os
onnx_path = list(uploaded.keys())[0]
print(f'Uploaded: {onnx_path}')

In [ ]:
import tensorrt as trt
import numpy as np
import time, os

OUTPUT_PATH = 'patchcore_cable.trt'
FP16 = True
print('TensorRT', trt.__version__)

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(TRT_LOGGER)
network = builder.create_network(0)  # explicit batch is the default in TensorRT 10
parser = trt.OnnxParser(network, TRT_LOGGER)

if not parser.parse_from_file(onnx_path):
    for i in range(parser.num_errors):
        print('Error:', parser.get_error(i))
    raise RuntimeError('ONNX parsing failed')

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
if FP16:
    config.set_flag(trt.BuilderFlag.FP16)
    print('FP16 enabled')

# anomalib exports a dynamic batch dim -> optimization profile is required
profile = builder.create_optimization_profile()
for i in range(network.num_inputs):
    inp = network.get_input(i)
    shape = (1, *inp.shape[1:])
    profile.set_shape(inp.name, shape, shape, shape)
config.add_optimization_profile(profile)

print('Building engine (2-5 min)...')
t0 = time.time()
serialized = builder.build_serialized_network(network, config)
if serialized is None:
    raise RuntimeError('Engine build failed')
print(f'Done in {time.time()-t0:.1f}s')

with open(OUTPUT_PATH, 'wb') as f:
    f.write(serialized)
print(f'Saved: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH)/(1<<20):.1f} MB)')

In [ ]:
import pycuda.driver as cuda
import pycuda.autoinit

runtime = trt.Runtime(TRT_LOGGER)
with open(OUTPUT_PATH, 'rb') as f:
    engine = runtime.deserialize_cuda_engine(f.read())
context = engine.create_execution_context()
stream = cuda.Stream()

names = [engine.get_tensor_name(i) for i in range(engine.num_io_tensors)]
inp = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
outs = [n for n in names if n != inp]
context.set_input_shape(inp, (1, *engine.get_tensor_shape(inp)[1:]))

bufs = {}
for n in names:
    shape = tuple(context.get_tensor_shape(n))
    h = cuda.pagelocked_empty(int(np.prod(shape)), trt.nptype(engine.get_tensor_dtype(n)))
    d = cuda.mem_alloc(h.nbytes)
    context.set_tensor_address(n, int(d))
    bufs[n] = (h, d)

dummy = np.random.rand(1, 3, 256, 256).astype(np.float32)

def infer():
    np.copyto(bufs[inp][0], dummy.ravel())
    cuda.memcpy_htod_async(bufs[inp][1], bufs[inp][0], stream)
    context.execute_async_v3(stream_handle=stream.handle)
    for n in outs:
        cuda.memcpy_dtoh_async(bufs[n][0], bufs[n][1], stream)
    stream.synchronize()

for _ in range(20): infer()  # warmup
latencies = []
for _ in range(200):
    t0 = time.perf_counter(); infer()
    latencies.append((time.perf_counter() - t0) * 1000)

print(f'Mean: {np.mean(latencies):.2f} ms | P50: {np.percentile(latencies,50):.2f} ms | P95: {np.percentile(latencies,95):.2f} ms | FPS: {1000/np.mean(latencies):.1f}')

In [ ]:
files.download(OUTPUT_PATH)